# Benchmark (vLLM): Anomaly Detection — Qwen3.5-4B

## 1. Install dependencies

In [ ]:
!pip install -q \
  "vllm==0.17.0" \
  "huggingface_hub>=0.30,<0.36" \
  pillow

!pip install -q --no-deps --upgrade "tokenizers>=0.22"
!pip install -q --no-deps "transformers==5.3.0"

!pip install -q --no-deps --force-reinstall "huggingface_hub>=1.3.0,<2.0"
!pip install -q --no-deps --force-reinstall "tokenizers==0.22.2"

import importlib.metadata as md
for pkg in ["vllm", "transformers", "tokenizers", "huggingface_hub"]:
    try:
        print(f"{pkg:18s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:18s} NOT INSTALLED")

## 2. (Optional) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. HuggingFace login

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN   = userdata.get('HF_TOKEN')
HF_REPO_ID = 'minsu0567/IAD-X1-GRPO-answer-last-no-hard'
login(token=HF_TOKEN)
print('HuggingFace login OK.')

## 4. GPU check

In [ ]:
!nvidia-smi

## 5. Build the shared 180-sample manifest

In [ ]:
import os

DRIVE_ROOT    = '/content/drive/MyDrive'
IAD_X1_DIR    = f'{DRIVE_ROOT}/IAD-X1'
EVAL_SRC      = f'{IAD_X1_DIR}/eval_src'
EVAL_DATA_DIR = f'{DRIVE_ROOT}/Uni-IAD_eval_dataset'
MANIFEST_JSON = f'{DRIVE_ROOT}/throughput_manifest_180.json'
BUILDER       = f'{EVAL_SRC}/build_throughput_manifest.py'

if os.path.isfile(MANIFEST_JSON):
    import json
    with open(MANIFEST_JSON) as f:
        _m = json.load(f)
    _benches = sorted({r['benchmark'] for r in _m})
    print(f'[OK] manifest exists: {len(_m)} samples, {len(_benches)} benchmarks -> {MANIFEST_JSON}')
    print(f'     {_benches}')
else:
    assert os.path.isfile(BUILDER), f'manifest builder not found: {BUILDER}'
    assert os.path.isdir(EVAL_DATA_DIR), f'eval dataset not found: {EVAL_DATA_DIR}'
    print(f'[..] manifest missing; building from {EVAL_DATA_DIR}')
    !python {BUILDER} --data_root {EVAL_DATA_DIR} --out {MANIFEST_JSON}

## 6. Run the benchmark via `!python`

In [ ]:
import os

MODEL_ID     = HF_REPO_ID
DEMO_REF     = '/content/000.png'
DEMO_QUERY   = '/content/025.png'
CSV_PATH     = '/content/batch_inference_benchmark_vllm_qwen35.csv'
BENCH_SCRIPT = f'{EVAL_SRC}/e2e_latency_qwen3_5.py'

assert os.path.isfile(BENCH_SCRIPT), f'benchmark script not found: {BENCH_SCRIPT}'
print('benchmark script ->', BENCH_SCRIPT)

!python {BENCH_SCRIPT} \
  --model-id "{MODEL_ID}" \
  --hf-token "{HF_TOKEN}" \
  --manifest-json "{MANIFEST_JSON}" \
  --demo-ref "{DEMO_REF}" \
  --demo-query "{DEMO_QUERY}" \
  --batch-size 1 \
  --warmup-batches 1 \
  --resize-to 512 \
  --max-model-len 32768 \
  --max-tokens 512 \
  --gpu-mem 0.9 \
  --csv "{CSV_PATH}"

## 7. (선택) 단일 (reference, query) 쌍 데모

In [ ]:
import os

REF   = '/content/000.png'
QUERY = '/content/025.png'

MODEL_ID     = HF_REPO_ID
BENCH_SCRIPT = f'{EVAL_SRC}/e2e_latency_qwen3_5.py'
assert os.path.isfile(BENCH_SCRIPT), f'benchmark script not found: {BENCH_SCRIPT}'

!python {BENCH_SCRIPT} \
  --model-id "{MODEL_ID}" \
  --hf-token "{HF_TOKEN}" \
  --demo-only \
  --demo-ref "{REF}" \
  --demo-query "{QUERY}" \
  --resize-to 512 \
  --max-model-len 32768 \
  --max-tokens 512 \
  --gpu-mem 0.9

## 8. Inspect results CSV

In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)
df